In [36]:

# import pandas as pd
# from sklearn.model_selection import train_test_split
# from sklearn.preprocessing import LabelEncoder, StandardScaler
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.metrics import accuracy_score, classification_report
# import re
# import joblib

# df = pd.read_csv("student.csv")

# # Drop unwanted columns
# df = df.drop(['id', 'City'], axis=1, errors='ignore')

# # ---------------------------
# # FIX 1: Clean Sleep Duration
# # ---------------------------
# def clean_sleep(x):
#     x = str(x).lower()

#     # Case: 5-6 hours
#     if "-" in x:
#         nums = re.findall(r'\d+', x)
#         if len(nums) == 2:
#             return (int(nums[0]) + int(nums[1])) / 2

#     # Case: less than 5 hours
#     if "less" in x:
#         return 4

#     # Case: more than 8 hours
#     if "more" in x:
#         return 9

#     # Case: single number inside string
#     nums = re.findall(r'\d+', x)
#     if len(nums) == 1:
#         return float(nums[0])

#     return None

# df['Sleep Duration'] = df['Sleep Duration'].apply(clean_sleep)
# df['Sleep Duration'] = df['Sleep Duration'].fillna(df['Sleep Duration'].median())

# # ----------------------------
# # FIX 2: Convert Financial Stress to numeric
# # ----------------------------
# df['Financial Stress'] = pd.to_numeric(df['Financial Stress'], errors='coerce')
# df['Financial Stress'] = df['Financial Stress'].fillna(df['Financial Stress'].median())

# # ----------------------------
# # Label Encoding for categorical columns
# # ----------------------------
# categorical_cols = [
#     'Gender',
#     'Profession',
#     'Dietary Habits',
#     'Degree',
#     'Have you ever had suicidal thoughts ?',
#     'Family History of Mental Illness'
# ]

# le = LabelEncoder()
# for col in categorical_cols:
#     df[col] = le.fit_transform(df[col].astype(str))

# # ----------------------------
# # Split features and target
# # ----------------------------
# X = df.drop('Depression', axis=1)
# y = df['Depression']

# # ----------------------------
# # Scale numeric features
# # ----------------------------
# scaler = StandardScaler()
# X = scaler.fit_transform(X)

# # ----------------------------
# # Train-test split
# # ----------------------------
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# # ----------------------------
# # Train Model
# # ----------------------------
# model = RandomForestClassifier()
# model.fit(X_train, y_train)

# # Predict
# y_pred = model.predict(X_test)

# # Evaluation
# print("Accuracy:", accuracy_score(y_test, y_pred))
# print(classification_report(y_test, y_pred))
# # Save model
# joblib.dump(model, "depression_model.pkl")

# # Save scaler
# joblib.dump(scaler, "scaler.pkl")
# ---------------------------
# IMPORTS
# ---------------------------
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import joblib

# ---------------------------
# LOAD DATA
# ---------------------------
df = pd.read_csv("student.csv")

# ---------------------------
# DROP UNWANTED COLUMNS
# ---------------------------
df = df.drop(['id', 'City'], axis=1, errors='ignore')

# ---------------------------
# FIX SLEEP DURATION COLUMN
# ---------------------------
def clean_sleep(x):
    x = str(x).lower()

    if "-" in x:   # 5-6 hours
        nums = re.findall(r'\d+', x)
        if len(nums) == 2:
            return (int(nums[0]) + int(nums[1])) / 2

    if "less" in x:    # less than 5 hours
        return 4

    if "more" in x:    # more than 8 hours
        return 9

    nums = re.findall(r'\d+', x)
    if len(nums) == 1:
        return float(nums[0])

    return np.nan

df["Sleep Duration"] = df["Sleep Duration"].apply(clean_sleep)
df["Sleep Duration"] = df["Sleep Duration"].fillna(df["Sleep Duration"].median())

# ---------------------------
# FIX FINANCIAL STRESS (STRING TO NUMBER)
# ---------------------------
df["Financial Stress"] = pd.to_numeric(df["Financial Stress"], errors="coerce")
df["Financial Stress"] = df["Financial Stress"].fillna(df["Financial Stress"].median())

# ---------------------------
# LABEL ENCODE CATEGORICAL COLUMNS
# ---------------------------
categorical_cols = [
    "Gender",
    "Profession",
    "Dietary Habits",
    "Degree",
    "Have you ever had suicidal thoughts ?",
    "Family History of Mental Illness"
]

le = LabelEncoder()
for col in categorical_cols:
    df[col] = le.fit_transform(df[col].astype(str))

# ---------------------------
# SPLIT FEATURES & TARGET
# ---------------------------
X = df.drop("Depression", axis=1)
y = df["Depression"]

# Save exact feature order
feature_order = list(X.columns)
joblib.dump(feature_order, "feature_order.pkl")

# ---------------------------
# SCALE FEATURES
# ---------------------------
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Save scaler
joblib.dump(scaler, "scaler.pkl")

# ---------------------------
# TRAIN-TEST SPLIT
# ---------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

# ---------------------------
# TRAIN MODEL
# ---------------------------
model = RandomForestClassifier()
model.fit(X_train, y_train)

# Save model
joblib.dump(model, "depression_model.pkl")

# ---------------------------
# PREDICT AND EVALUATE
# ---------------------------
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

print("\nModel, Scaler, and Feature Order saved successfully!")



Accuracy: 0.8274502777280057

Classification Report:
               precision    recall  f1-score   support

           0       0.80      0.78      0.79      2343
           1       0.84      0.86      0.85      3238

    accuracy                           0.83      5581
   macro avg       0.82      0.82      0.82      5581
weighted avg       0.83      0.83      0.83      5581


Model, Scaler, and Feature Order saved successfully!


In [42]:

import pandas as pd
import joblib

# ---------------------------
# LOAD MODEL, SCALER, FEATURE ORDER
# ---------------------------
model = joblib.load("depression_model.pkl")
scaler = joblib.load("scaler.pkl")
feature_order = joblib.load("feature_order.pkl")

print("Model, scaler, and feature order loaded successfully!")

# ---------------------------
# NEW STUDENT INPUT
# ---------------------------
new_data = pd.DataFrame([{
    "Gender": 1,
    "Age": 24.0,
    "Profession": 0,
    "Academic Pressure": 3.0,
    "CGPA": 6.1,
    "Study Satisfaction": 3.0,
    "Job Satisfaction": 0.0,
    "Sleep Duration": 5.5,
    "Dietary Habits": 1,
    "Degree": 4,
    "Have you ever had suicidal thoughts ?": 1,
    "Work/Study Hours": 11.0,
    "Financial Stress": 1.0,
    "Family History of Mental Illness": 1,
    "Work Pressure": 0.0
}])


# ---------------------------
# MATCH TRAINING FEATURE ORDER
# ---------------------------
new_data = new_data[feature_order]

# ---------------------------
# SCALE INPUT
# ---------------------------
new_scaled = scaler.transform(new_data)

# ---------------------------
# PREDICTION + PROBABILITY
# ---------------------------
pred = model.predict(new_scaled)[0]
prob = model.predict_proba(new_scaled)[0]

print("Predicted Depression Status:", pred)
print("Probability of Class 0 (No Depression):", prob[0])
print("Probability of Class 1 (Depression):", prob[1])

# More user-friendly output
if pred == 1:
    print(f"→ The student is predicted to have depression with probability {prob[1]:.2f}")
else:
    print(f"→ The student is predicted NOT to have depression with probability {prob[0]:.2f}")


Model, scaler, and feature order loaded successfully!
Predicted Depression Status: 1
Probability of Class 0 (No Depression): 0.28
Probability of Class 1 (Depression): 0.72
→ The student is predicted to have depression with probability 0.72
